In [8]:
import os

def convert_tflite_to_cpp(tflite_model_path, output_name="model_data"):
    if not os.path.exists(tflite_model_path):
        print(f"Error: File {tflite_model_path} không tồn tại!")
        return

    with open(tflite_model_path, 'rb') as f:
        model_content = f.read()

    # Tạo nội dung cho file .cc (Chứa dữ liệu thực)
    hex_array = [f'0x{byte:02x}' for byte in model_content]
    lines = []
    for i in range(0, len(hex_array), 12):
        lines.append("    " + ", ".join(hex_array[i:i+12]))
    
    # Thêm căn lề 16 byte để ESP32-S3 chạy nhanh hơn
    cc_content = f'#include "{output_name}.h"\n\n'
    cc_content += f'const unsigned char g_model[] __attribute__((aligned(16))) = {{\n'
    cc_content += ",\n".join(lines)
    cc_content += f'\n}};\n'
    cc_content += f'const unsigned int g_model_len = {len(model_content)};\n'

    with open(f"{output_name}.cc", 'w') as f:
        f.write(cc_content)

    # Tạo nội dung cho file .h (Chỉ khai báo)
    h_content = f"""#ifndef {output_name.upper()}_H
#define {output_name.upper()}_H

extern const unsigned char g_model[];
extern const unsigned int g_model_len;

#endif"""

    with open(f"{output_name}.h", 'w') as f:
        f.write(h_content)

    print(f"✅ Đã tạo xong {output_name}.h và {output_name}.cc tại thư mục hiện tại!")

# Chạy trên Jupyter:
convert_tflite_to_cpp("traffic_sign_final_int8.tflite")

✅ Đã tạo xong model_data.h và model_data.cc tại thư mục hiện tại!
